# Exercise 5.4 — Random forests

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day5/day5-04-random-forest.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

A single decision tree is easy to read but unstable. A few different people in
the training rows can give a very different tree. A **random forest** grows
many trees and lets them vote. Two things make the trees differ from each
other:

- each tree is grown on a **bootstrap sample** of the training rows: as many
  rows as the training set, drawn at random with replacement;
- at each split, each tree may only try a **random subset of the predictors**.

The forest's prediction is the average of the trees' votes. The errors of the
single trees partly cancel out.

**Goal.** In this exercise you will:

- tune a random forest with a workflow and 5-fold cross-validation, as you did
  for the tree in Exercise 5.3;
- fit the final forest and evaluate it once on the test rows;
- compare it with your tuned tree.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas matplotlib scikit-learn

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

The same 750 people and five predictors as Exercise 5.3, from
[`data/diabetes_clean.csv`](data/diabetes_clean.csv). The outcome is 1 for `pos`
and 0 for `neg`.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["pregnant", "glucose", "mass", "pedigree", "age", "diabetes"]]
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

predictors = ["pregnant", "glucose", "mass", "pedigree", "age"]

The same split as before: 600 people for training and 150 for testing.

In [ ]:
train, test = train_test_split(diabetes, test_size=0.2, random_state=42)

------------------------------------------------------------------------

### Part 1 — From one tree to a forest

`RandomForestClassifier()` with 500 trees. `random_state` makes the forest the
same every time, and `n_jobs=-1` grows the trees on all the computer's cores.
Two hyperparameters will be tuned:

- `max_features`: how many predictors each split may choose from (1 to 5 here),
  called `mtry` in R;
- `min_samples_split`: the smallest node that may still be split, as for the
  tree.

Compared with Exercise 5.3, only the model line has changed.

In [ ]:
rf = RandomForestClassifier(n_estimators=500, random_state=123, n_jobs=-1)

> **Learning points**
>
> - A random forest is many trees, each grown on a bootstrap sample of the rows and trying a random subset of predictors at each split. Their votes are averaged.
> - A small `max_features` makes the trees more different from each other.

> **Where to find a model's hyperparameters**
>
> Each scikit-learn model's page lists all its settings. For this forest
> see [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html);
> for any other model, start from the
> [API reference](https://scikit-learn.org/stable/api/index.html).

------------------------------------------------------------------------

### Part 2 — Tuning `max_features` and `min_samples_split`

Five values of each, 25 combinations. Each one is judged by 5-fold
cross-validation on the training rows, as in Exercise 5.3. That is 125 forests.

In [ ]:
rf_grid = {"max_features":      [1, 2, 3, 4, 5],
           "min_samples_split": [2, 4, 6, 8, 10]}

rf_search = GridSearchCV(rf, param_grid=rf_grid, cv=5, scoring="roc_auc")
rf_search = rf_search.fit(train[predictors], train["diabetes"])

rf_results = pd.DataFrame(rf_search.cv_results_)
rf_results = rf_results[["param_max_features", "param_min_samples_split",
                         "mean_test_score", "std_test_score"]]

fig, ax = plt.subplots()
for n in rf_grid["min_samples_split"]:
    rows = rf_results[rf_results["param_min_samples_split"] == n]
    ax.plot(rows["param_max_features"], rows["mean_test_score"], marker="o", label=str(n))
ax.set_xlabel("max_features (mtry)")
ax.set_ylabel("Mean cross-validated AUC")
ax.legend(title="min_samples_split")
plt.show()

Now show the five best combinations by AUC:

In [ ]:
# Replace every ______ below, then run this cell.
rf_results.sort_values(______, ascending=False).head(5).round(3)

> **Stuck?**
>
> Sort by the column that holds the mean cross-validated AUC, written in quotes.

**Question!** Which values of `max_features` and `min_samples_split` work
best? Are the differences large compared with the spread across folds?

> **Learning points**
>
> - Tuning a forest uses the same pipeline, grid and folds as tuning a tree.
> - Look at the whole grid, not only the winner: many combinations are close.

------------------------------------------------------------------------

### Part 3 — The final forest, evaluated once

`GridSearchCV()` has already refitted the best combination on all 600 training
rows:

In [ ]:
print(rf_search.best_params_)
final_rf = rf_search.best_estimator_

test_class = final_rf.predict(test[predictors])
test_prob  = final_rf.predict_proba(test[predictors])[:, 1]

Now evaluate it on the test rows. Sensitivity and specificity are as in
Exercise 4.3; in scikit-learn the specificity is the recall of class 0:

In [ ]:
# Replace every ______ below, then run this cell.
truth, predicted = test["diabetes"], ______
print("test accuracy:   ", round(accuracy_score(truth, predicted), 3))
print("test sensitivity:", round(recall_score(truth, predicted), 3))
print("test specificity:", round(recall_score(truth, predicted, pos_label=0), 3))
print("test AUC:        ", round(roc_auc_score(truth, ______), 3))

> **Stuck?**
>
> Accuracy, sensitivity and specificity need the predicted classes,
> `test_class`. The AUC needs the predicted probabilities, `test_prob`.

**Question!** Look back at your tuned tree in Exercise 5.3. Is the forest
better on the test rows? Is it worth losing a single tree that you can read?

> **Learning points**
>
> - Fit the final forest once on all the training rows, then use the test rows once.
> - A forest is usually more accurate than a single tree, but it cannot be read as a picture.